In [0]:
from pyspark.sql.functions import (
    col, current_timestamp, lit, to_timestamp,
    round, when, lag, abs
)
from pyspark.sql.window import Window

# Storage config
storage_account = "theportfoliostorage"
container = "datalake"

spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net", 
               "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", 
               dbutils.secrets.get(scope="de-portfolio-scope", key="sp-client-id"))
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", 
               dbutils.secrets.get(scope="de-portfolio-scope", key="sp-client-secret"))
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net", 
               f"https://login.microsoftonline.com/{dbutils.secrets.get(scope='de-portfolio-scope', key='sp-tenant-id')}/oauth2/token")

# Read Bronze
df_bronze = spark.table("crypto.bronze.prices_raw")

print(f"Bronze row count: {df_bronze.count()}")
df_bronze.printSchema()

In [0]:
when(abs(col("batch_price_change_pct")) > 0.05, lit(True))

In [0]:
# Step 1: Parse last_updated from string to proper timestamp
# Step 2: Round float columns to sensible precision
# Step 3: Add price movement indicators between batches

# Define window for lag calculations - ordered by batch per coin
coin_window = Window.partitionBy("symbol").orderBy("_batch_id")

df_silver = df_bronze \
    .withColumn("last_updated", 
        to_timestamp(col("last_updated"), "yyyy-MM-dd'T'HH:mm:ss.SSS'Z'")) \
    .withColumn("current_price", round(col("current_price"), 2)) \
    .withColumn("price_change_24h", round(col("price_change_24h"), 4)) \
    .withColumn("price_change_percentage_24h", 
        round(col("price_change_percentage_24h"), 4)) \
    .withColumn("high_24h", round(col("high_24h"), 2)) \
    .withColumn("low_24h", round(col("low_24h"), 2)) \
    .withColumn("previous_price", 
        lag("current_price", 1).over(coin_window)) \
    .withColumn("batch_price_change",
        round(col("current_price") - col("previous_price"), 2)) \
    .withColumn("batch_price_change_pct",
        round((col("current_price") - col("previous_price")) 
              / col("previous_price") * 100, 4)) \
    .withColumn("price_direction",
        when(col("batch_price_change") > 0, lit("UP"))
        .when(col("batch_price_change") < 0, lit("DOWN"))
        .otherwise(lit("FLAT"))) \
    .withColumn("is_volatile",
        when(abs(col("batch_price_change_pct")) > 0.5, lit(True))
        .otherwise(lit(False))) \
    .withColumn("_silver_processed_at", current_timestamp())

print(f"Silver row count: {df_silver.count()}")
df_silver.printSchema()

In [0]:
# Check price direction distribution across all coins
print("Price direction distribution:")
df_silver.groupBy("price_direction") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()

# Check volatility
print("Volatile batches:")
df_silver.filter(col("is_volatile") == True) \
    .select("name", "symbol", "_batch_id", 
            "current_price", "batch_price_change_pct") \
    .orderBy("batch_price_change_pct") \
    .show(truncate=False)

# Check nulls in previous_price 
# First batch per coin will always be null - expected behaviour
print("Null previous_price count (should be 10 - one per coin):")
df_silver.filter(col("previous_price").isNull()).count()

In [0]:
# Write to Silver Delta table
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("crypto.silver.prices_clean")

print("Silver table written successfully")

# Verify
spark.sql("""
    SELECT 
        symbol,
        COUNT(*) as batches,
        MIN(current_price) as min_price,
        MAX(current_price) as max_price,
        MAX(current_price) - MIN(current_price) as price_range
    FROM crypto.silver.prices_clean
    GROUP BY symbol
    ORDER BY price_range DESC
""").show(truncate=False)